# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Msdff/FlyRankAiAssignment/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


**Finding #4:- The Freshness Multiplier**

The paper says that old pages (365+ days old) that were updated recently got much better results.
It says
Health increased 3.2 times
Impressions increased 57 times
But My question is How many pages were used to get this result?
1. I want to know if many pages improved, or if only a few pages had a very big improvement. For example, the impressions went from 71 to 4039. Maybe this big change happened because of only a few pages. So, I want to check if this result is true for many pages or only a few pages.

**Finding #7:- The Winning Combinations**
The paper says that pages with transactional intent and low competition perform the best.
My question: Did the researchers also consider the age of the page and how much content it has?
Maybe these pages performed better because they were already older and had better or more content.
The paper only compares the pages at one point in time. It does not compare their performance before and after.
So, we cannot be sure that transactional intent + low competition was the main reason for their better performance.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
import os
print(os.getcwd())
os.chdir("..")   

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

features = ["days_since_last_update", "impressions_90d", "avg_position", "word_count", "search_volume"]

print(df.shape)
print(features)

(30000, 45)
['days_since_last_update', 'impressions_90d', 'avg_position', 'word_count', 'search_volume']


In [4]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import numpy as np


X = df[features].fillna(0)
y = df["is_declining_label"]

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_naive = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model_naive.fit(X_train_naive, y_train_naive)

test_probs_naive = model_naive.predict_proba(X_test_naive)[:, 1]

for k in (20, 50):
    print(f"NAIVE random split — Precision@{k}:", round(precision_at_k(test_probs_naive, y_test_naive, k), 3))

NAIVE random split — Precision@20: 0.75
NAIVE random split — Precision@50: 0.86


**BEFORE Result**
The model got:
* Precision@20 = 75%
* Precision@50 = 86%
So, at first, the model looks very good.
But there is a problem. The same client can have pages in both training and testing.
For example: Client A -> Training: some pages    Testing: some pages
So, the model may already know the client's pattern. It may not be learning a pattern that works for a new client. So, we should not fully trust these results yet.

In [ ]:

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

X_train, y_train = train_df[features].fillna(0), train_df["is_declining_label"]
X_test, y_test = test_df[features].fillna(0), test_df["is_declining_label"]

model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)
test_df["model_prob"] = model.predict_proba(X_test)[:, 1]

for k in (20, 50):
    print(f"HONEST client-grouped split — Precision@{k}:", round(precision_at_k(test_df["model_prob"], y_test, k), 3))

HONEST client-grouped split — Precision@20: 0.65
HONEST client-grouped split — Precision@50: 0.62


**AFTER Result**
The model got:
Precision@20 = 65%
Precision@50 = 62%
Now we compare it with the BEFORE result:
Split type	                      Precision@20	Precision@50
Naive random (before)                  75%	        86%
Honest client-grouped (after)          65%	        62%


The before result looked better, but it was not a fair test.The after test is more honest because the model was tested on clients it had not seen before.So, the model is actually not as good as the first test made it look. The biggest difference is at Top 50: 86% → 62% .This shows that how we test the model is very important.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

This is a new leakage check done in Week 6 for the same 5 features used in the Week-5 model.

days_since_last_update = We know when the page was last updated.
impressions_90d = We know how many times the page was shown.
avg_position = We know the page's average search position.
word_count = We know how many words are on the page.
search_volume = We know how many people search for the keyword.

None of these features use the label or future information. trend_direction was used to make the label, but it was not used as a feature. We also did not use health_score or other FlyRank features because they were not in this dataset. So, there was no data leakage in the Week-5 model.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

I checked my earlier claims from Weeks 1, 4, and 5 to see if I said anything too strongly.
**Week 1:**
This project can produce observed and directional findings. The results show patterns in the data, but they do not guarantee what will happen. All recommendations are only meant to help humans make decisions, not to replace human judgment.
I already wrote this in a careful and safe way. I clearly explained what my project can say and what it cannot say. I also did not say that my results were definitely proven or that one thing caused another. So, I do not need to change this claim. I want the rest of my project to follow the same careful style.

**Week 4**
This may show a real pattern, but it is only an observed pattern and is not confirmed.
This was about most of the top-20 pages belonging to one client. I already explained this carefully. I also mentioned another possible reason: that client may simply have more content or pages than the other clients. So, I did not immediately say that one thing caused the other. This claim also does not need to be changed.

**Week 5**
This helps explain why the model did worse than the baseline. It is approaching the problem in a very different way than the baseline. This claim sounded too certain, so I changed it.

New claim:
The model's low use of days_since_last_update is one observed factor that may help explain why its measured precision was lower than the baseline.
But this is only a possible explanation, not a proven cause.

There may be other reasons too. For example:

We only used five features.
We used only one train/test split.
The features may not have enough information 
The test had only a small number of clients.
I have not tested each of these reasons separately, so I cannot say which one is the real cause.

**Overall**
After checking Weeks 1, 4, and 5, I found that most of my claims were already written carefully, especially in Weeks 1 and 4. The main claim I changed was the Week-5 explanation for why the model performed worse. The old claim sounded like it was saying one thing caused the model to perform worse. But my evidence came from only one train/test split, so I cannot be that sure.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.